# Multi-Modal LLM Style Analysis: run it in Google Colab

This notebook runs our Phase B research pipeline (Braude College capstone project 26-2-R-6) in the
browser. Nothing is installed on your own computer.

**To run it:** Runtime → Run all. When Colab asks for access to Google Drive, allow it.

| Part | What it shows | Needs a Gemini key? |
|---|---|---|
| 1. Setup | Downloads our code from GitHub and installs it | No |
| 2. The corpus | The 49 creations and the integrity check | No |
| 3. The whole method on a tiny example | All six steps, end to end, with a stand-in model | No |
| 4. The method on real text | Gemini rewrites passages of *Macbeth* and *Henry VIII*; each step's output | Yes |
| 5. Run it again | The same numbers from saved answers, with no new calls | Yes |
| 6. The proof of concept | The full method on five creations, with its cost estimated first | A paid key |
| 7. Is Drive fast enough? | An optional timing check of the folder that keeps the saved answers | No |

**The Gemini key** (Parts 4 to 6): click the key icon on the left (Secrets) → Add new secret →
name `GEMINI_API_KEY`, paste the key, and switch on *Notebook access*. Without a key, Parts 4 to 6
are skipped and everything else still runs.

Results are saved to your Google Drive, in the folder `MyDrive/mmlsa`, so nothing is lost when Colab
disconnects.

## Part 1. Setup

The settings below are the only things you may want to change. Part 4 rewrites `CHUNKS` passages and
makes one profile call, so it uses `CHUNKS + 1` calls of the free tier's 20 a day.

With a paid key, set `TIER = "paid"`: every live command then uses the paid request limits instead of
pacing itself to the free tier. Part 6 spends about a thousand calls, so it also waits for
`RUN_POC = True`.

In [ ]:
MODEL = "gemini-3.1-flash-lite"  # the model we recommend for the runs (docs/DECISIONS.md I28)
PLAYS = ["the_tragedy_of_macbeth", "king_henry_viii"]  # one expected authentic, one co-written
CHUNKS = 6  # passages Gemini rewrites in Part 4
TIER = "free"  # "free" paces the calls to the free tier's limits; "paid" with a paid key
RUN_POC = False  # Part 6 spends about 1,000 paid calls; set to True to run it
POC_RUN_ID = None  # to continue a Part 6 run that Colab cut off, its run id, as a string
CACHE_ON_DRIVE = True  # False: saved answers on the Colab disk, copied to Drive after each part
REPO_URL = "https://github.com/eliranmeleh/Multi-Modal-LLM-Style-Analysis.git"

# Request limits for every live command. Free: measured on 2026-08-29 and 2026-11-21
# (docs/RESULTS.md F-12); our rate limiter estimates tokens from words, which undercounts, so the
# token budget is set well under the real 250,000. Paid: check your project's limits in AI Studio.
LIMITS = {
    "free": {
        "llm.requests_per_minute": 4,
        "llm.concurrency": 1,
        "llm.tokens_per_minute": 100000,
        "llm.max_transient_retries": 8,
    },
    "paid": {
        "llm.requests_per_minute": 240,
        "llm.concurrency": 8,
        "llm.tokens_per_minute": 1000000,
        "llm.max_transient_retries": 5,
    },
}

# Dollars per million tokens (input, output), from Google's pricing page on 2026-11-23. Used only
# to print estimates; a model missing here is run as usual, without a cost estimate.
PRICES = {
    "gemini-3.1-flash-lite": (0.25, 1.50),
    "gemini-3.5-flash-lite": (0.30, 2.50),
    "gemini-3.8-flash": (0.75, 3.75),
}

Download the code and install it. In Colab this takes about a minute. If the code is already there
from an earlier run in the same session, it is updated to the latest version first. On a laptop, run
the notebook from inside the repository and this step only finds the code.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if TIER not in LIMITS:
    raise ValueError(f"TIER must be one of {sorted(LIMITS)}, not {TIER!r}")


def run_quietly(command):
    # Run a setup command and show its output only if it fails.
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        print(result.stdout, result.stderr)
        raise RuntimeError("setup step failed: " + " ".join(command))
    return result.stdout.strip()


if IN_COLAB:
    REPO_DIR = Path("/content/Multi-Modal-LLM-Style-Analysis")
    if REPO_DIR.exists():
        run_quietly(["git", "-C", str(REPO_DIR), "pull", "--quiet", "--ff-only"])
    else:
        run_quietly(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    run_quietly([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[gemini]"])
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))  # so this notebook can import our code directly
importlib.invalidate_caches()
COMMIT = run_quietly(["git", "rev-parse", "--short", "HEAD"])
print("Our code is ready in", REPO_DIR, "at commit", COMMIT)

Connect Google Drive, where the results are kept, and read the Gemini key from Colab Secrets.

The saved model answers (the cache) hold one small file per call, and the full corpus run alone
makes about 9,000. If Part 7 shows that Drive is slow with that many files, set
`CACHE_ON_DRIVE = False`: the answers are then kept on the Colab disk and copied to Drive at the end
of each live part. The cost is that answers from a part that Colab cut off are not copied, and are
paid for again next time.

In [ ]:
import shutil

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/mmlsa")
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:  # no such secret, or Notebook access is switched off
        print("No GEMINI_API_KEY in Colab Secrets, so Parts 4 to 6 will be skipped.")
else:
    SAVE_DIR = Path("runs/notebook")  # inside the repository, ignored by git

SAVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE = SAVE_DIR / "cache"
CACHE_DIR = DRIVE_CACHE if CACHE_ON_DRIVE or not IN_COLAB else Path("/content/mmlsa-cache")


def copy_new_files(source, target):
    # Copy the files under source that target does not have yet; return how many were copied.
    copied = 0
    for path in source.rglob("*.json") if source.exists() else []:
        destination = target / path.relative_to(source)
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, destination)
            copied += 1
    return copied


def save_cache():
    # With the cache on the Colab disk, copy the answers saved since the last copy to Drive.
    if CACHE_DIR != DRIVE_CACHE:
        print(f"Copied {copy_new_files(CACHE_DIR, DRIVE_CACHE)} new saved answers to Drive")


if CACHE_DIR != DRIVE_CACHE:
    print(f"Copied {copy_new_files(DRIVE_CACHE, CACHE_DIR)} saved answers from Drive to Colab")

HAVE_KEY = bool(os.environ.get("GEMINI_API_KEY")) or (not IN_COLAB and Path(".env").exists())
print("Results are saved in", SAVE_DIR.resolve())
print("Saved answers are kept in", CACHE_DIR.resolve())
print("Gemini key:", "found" if HAVE_KEY else "not found", f"(tier: {TIER})")

Every step below runs our command-line tool, `python -m mmlsa`, exactly as on a laptop. This small
helper runs it, prints what it says, and ends with how many model answers were new and how many came
from saved copies. After each live part, `cost_report` adds up the tokens of the new answers from the
run's call log and prices them with `PRICES`.

In [ ]:
import json


def mmlsa(*args):
    # Run one mmlsa command; return the lines it printed for people.
    command = [sys.executable, "-m", "mmlsa", *args]
    environment = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    calls, printed = None, []
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=environment,
    ) as process:
        for line in process.stdout:
            try:
                event = json.loads(line) if line.startswith("{") else None
            except json.JSONDecodeError:
                event = None
            routine = line.startswith("AFC is enabled") or (
                line.startswith("HTTP Request:") and "200 OK" in line
            )
            if event is None and not routine:
                print(line, end="")  # what the tool says to people, and any error from Gemini
                printed.append(line.rstrip("\n"))
            elif event is not None and event.get("event") == "runner.complete":
                calls = event  # its detailed log is machine-readable; keep only the totals
    if calls:
        print(f"\nModel answers: {calls['live']} new, {calls['cached']} from saved copies")
    if process.returncode != 0:
        raise RuntimeError(f"mmlsa {args[0]} stopped with exit code {process.returncode}")
    return printed


def limit_settings():
    # The request limits of the chosen TIER, as --set options for a live command.
    options = []
    for key, value in LIMITS[TIER].items():
        options += ["--set", f"{key}={value}"]
    return options


def dollars(input_tokens, output_tokens, model=MODEL):
    # What the tokens cost at the model's PRICES, or None if the model has no price.
    if model not in PRICES:
        return None
    price_in, price_out = PRICES[model]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def cost_report(calls_file, model=MODEL):
    # Add up the tokens of the new (not saved) answers in a call log, and what they cost.
    entries = [
        json.loads(line)
        for line in Path(calls_file).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    new = [entry for entry in entries if not entry.get("cached")]
    input_tokens = sum(entry.get("input_tokens", 0) for entry in new)
    output_tokens = sum(entry.get("output_tokens", 0) for entry in new)
    print(f"Tokens of the {len(new)} new answers: {input_tokens:,} in, {output_tokens:,} out")
    cost = dollars(input_tokens, output_tokens, model)
    if cost is None:
        print(f"No price for {model} in PRICES, so no cost estimate")
    else:
        note = " (free on the free tier; a paid key would pay this)" if TIER == "free" else ""
        print(f"Cost at {model} prices: ${cost:.2f}{note}")

## Part 2. The corpus

The input is the 49 creations attributed to Shakespeare, taken from Project Gutenberg and cleaned
(headers, licences and cast lists removed). The table lists them by length. The check after it
recomputes every file's fingerprint and confirms that nothing changed since we built the corpus.

In [ ]:
import pandas as pd
from IPython.display import display

manifest = json.loads(Path("data/manifest.json").read_text(encoding="utf-8"))
corpus = (
    pd.DataFrame(manifest["sets"]["texts"])[["title", "n_words"]]
    .rename(columns={"title": "creation", "n_words": "words"})
    .sort_values("words", ascending=False)
    .reset_index(drop=True)
)
print(f"{len(corpus)} creations, {corpus['words'].sum():,} words")
display(corpus)

In [ ]:
mmlsa("corpus", "verify")

## Part 3. The whole method on a tiny example

Three short texts and a **stand-in model** that edits text by fixed rules instead of a real AI. The
numbers mean nothing; the point is to watch all six steps run in seconds and to see what the final
output looks like: one score per text, a threshold, and the chart we will draw for all 49 creations.

In [ ]:
mmlsa(
    "run",
    "--config",
    "configs/mini.yaml",
    "--set",
    f"run.out_dir={(SAVE_DIR / 'runs').as_posix()}",
    "--set",
    f"llm.cache_dir={CACHE_DIR.as_posix()}",
)


def newest_run(tag):
    # The most recent run directory with this tag.
    return max((SAVE_DIR / "runs").glob(f"*-{tag}"), key=lambda path: path.stat().st_mtime)


mini_run = newest_run("mini")

**Steps 5 and 6.** `score_mean` is each text's average style distance over its passages and runs
(Step 5). Otsu's method then places the threshold `tau` where it best splits the scores into two
groups (Step 6). Texts above it are labelled *suspicious*.

In [ ]:
from IPython.display import Image


def show_run(run_dir):
    # A run's scores, its threshold, and the sorted scatter of the scores.
    scores = pd.read_csv(run_dir / "scores.csv")
    display(scores[["title", "n_chunks", "score_mean", "score_std", "label"]])
    threshold = json.loads((run_dir / "threshold.json").read_text(encoding="utf-8"))
    print(f"Threshold tau = {threshold['tau']:.4f} ({threshold['method']})")
    print("Suspicious:", ", ".join(threshold["suspicious"]) or "none")
    display(Image(filename=str(run_dir / "figures" / "sorted_scatter.png")))
    return scores


show_run(mini_run);

## Part 4. The method on real text, step by step

Gemini reads *Macbeth* and *Henry VIII* and writes a style profile (Step 1). The plays are cut into
400-word passages (Step 2), and Gemini rewrites a few of them in the profile's style (Step 3). Then we
measure how much each rewrite changed (Step 4). This part needs the Gemini key. On the free tier it
takes a few minutes, because the free tier allows four calls and 250,000 input tokens a minute, and
the profile call alone sends both plays, about 70,000 tokens. Lines that say `503` or `429` mean
Gemini was busy; the tool waits and tries again by itself.

This part uses `compare`, our model-comparison tool, because it shows each step's output for a few
passages. Part 6 runs the pipeline itself.

In [ ]:
import time

COMPARE = [
    "compare",
    "--config",
    "configs/default.yaml",
    "--model",
    f"gemini:{MODEL}",
    "--chunks",
    str(CHUNKS),
    "--set",
    f"corpus.include_ids=[{','.join(PLAYS)}]",
    "--set",
    f"llm.cache_dir={CACHE_DIR.as_posix()}",
    *limit_settings(),
    "--yes",
]

live_dir = SAVE_DIR / "live" / time.strftime("%Y%m%d-%H%M%S")
if HAVE_KEY:
    try:
        mmlsa(*COMPARE, "--out-dir", live_dir.as_posix())
    finally:
        save_cache()
else:
    print("Skipped: no Gemini key. Add GEMINI_API_KEY in Colab Secrets and run this cell again.")

LIVE_OK = any((live_dir / "profiles").glob("*.json"))
if LIVE_OK:
    cost_report(live_dir / "calls.jsonl")
elif HAVE_KEY:
    print("\nGemini did not complete the profile this time; the free tier is sometimes busy.")
    print("Wait a minute and run this cell again. The cells below are skipped until it works.")

### Step 1. The style profile

What Gemini says the common style of the two plays is. The prompt never names the author.

In [ ]:
import textwrap

if LIVE_OK:
    profile_file = next((live_dir / "profiles").glob("*.json"))
    profile = json.loads(profile_file.read_text(encoding="utf-8"))["profile"]
    for name, text in (profile.get("fields") or {"profile": profile.get("text", "")}).items():
        print(name.replace("_", " ").upper())
        print(textwrap.fill(str(text), 100), end="\n\n")

### Steps 2 and 3. A passage and its rewrite

Step 2 cut the plays into 400-word passages; on the left is one of them. On the right is Gemini's
rewrite of it (Step 3). Highlighted are the **function words** (*the, of, and, thou, you, ...*) that
the rewrite changed: yellow in the original, green in the rewrite. Change `PASSAGE` to see another
one; it counts only the rewrites that passed the validation checks.

In [ ]:
import html
import re

import Levenshtein
from IPython.display import HTML

from mmlsa.distance.fwed import FunctionWordEditDistance, encode_sequence
from mmlsa.distance.tokenize import (
    EDGE_PUNCTUATION,
    extract_function_words,
    load_function_words,
    normalize_for_tokenizing,
)

PASSAGE = 0  # 0 up to the number of rewrites that passed - 1
FUNCTION_WORDS_FILE = "data/function_words/en_core_v1.txt"
FUNCTION_WORDS = load_function_words(FUNCTION_WORDS_FILE)
FUNCTION_WORD_SET = frozenset(FUNCTION_WORDS)


def read_passage(path):
    # A side-by-side file holds the original passage, then the model's cleaned rewrite.
    sections = re.split(r"^## ", path.read_text(encoding="utf-8"), flags=re.MULTILINE)
    original = sections[1].removeprefix("original").strip()
    header, _, rewrite = sections[2].partition("\n")
    return original, rewrite.strip(), header.strip()


def changed_positions(original, rewrite):
    # The positions FWED counts as changed, from the same alignment the metric uses.
    before = extract_function_words(original, FUNCTION_WORDS)
    after = extract_function_words(rewrite, FUNCTION_WORDS)
    operations = Levenshtein.editops(
        encode_sequence(before, FUNCTION_WORDS), encode_sequence(after, FUNCTION_WORDS)
    )
    removed = {source for kind, source, _ in operations if kind in ("replace", "delete")}
    added = {target for kind, _, target in operations if kind in ("replace", "insert")}
    return removed, added


def marked(text, positions, colour):
    # The passage word by word, with the function words at the given positions highlighted.
    words, index = [], 0
    for word in normalize_for_tokenizing(text).split():
        shown = html.escape(word)
        if word.casefold().strip(EDGE_PUNCTUATION) in FUNCTION_WORD_SET:
            if index in positions:
                shown = f'<mark style="background:{colour};color:black">{shown}</mark>'
            index += 1
        words.append(shown)
    return " ".join(words)


if LIVE_OK:
    comparison = pd.read_csv(live_dir / "comparison.csv")
    passed = comparison[comparison["ok"]]  # only rewrites that passed validation
    shown = min(max(PASSAGE, 0), len(passed) - 1)
    if shown != PASSAGE:
        print(f"PASSAGE = {PASSAGE}, but {len(passed)} rewrites passed; showing passage {shown}.")
    chosen = passed.iloc[shown]
    passage_file = live_dir / "rewrites" / f"{chosen.creation_id}_{chosen.chunk_index:04d}.md"
    original, rewrite, header = read_passage(passage_file)
    removed, added = changed_positions(original, rewrite)
    cell = "vertical-align:top;text-align:left;line-height:1.5"
    display(
        HTML(
            f"<p><b>{passage_file.stem}</b>: {html.escape(header)}</p>"
            "<table style='width:100%;table-layout:fixed'><tr>"
            f"<th style='{cell}'>Original passage (Step 2)</th>"
            f"<th style='{cell}'>Gemini's rewrite (Step 3)</th></tr>"
            f"<tr><td style='{cell}'>{marked(original, removed, '#ffe08a')}</td>"
            f"<td style='{cell}'>{marked(rewrite, added, '#b6f0b6')}</td></tr></table>"
        )
    )

### Step 4. The style distance (FWED)

Take the function words of each passage, in order. FWED is the number of edits needed to turn one
sequence into the other, divided by the length of the longer sequence. 0 means no change; a passage
already in the corpus style should score low.

In [ ]:
if LIVE_OK:
    result = FunctionWordEditDistance(FUNCTION_WORDS_FILE, with_detail=True)(original, rewrite)
    edits = result.detail["edits"]
    longer = max(result.n_units_original, result.n_units_rewrite)
    print(
        f"Function words: {result.n_units_original} in the original, "
        f"{result.n_units_rewrite} in the rewrite"
    )
    for edit in edits:
        if edit["op"] == "replace":
            print(f"  replace '{edit['from']}' with '{edit['to']}'")
        elif edit["op"] == "delete":
            print(f"  remove  '{edit['from']}'")
        else:
            print(f"  add     '{edit['to']}'")
    print(f"FWED = {len(edits)} edits / {longer} = {result.value:.4f}")
    print(f"The pipeline recorded {chosen.delta:.4f} for this passage in Part 4: the same number.")

### Steps 5 and 6, as a preview

Step 5 averages each play's passages. Step 6, the threshold, needs the whole corpus, so on two plays
and a handful of passages this is only a preview of the shape of the result; Part 3 showed the real
mechanism, and the full run (experiment E4) uses every passage of all 49 creations.

In [ ]:
if LIVE_OK:
    usable = comparison[comparison["ok"]]
    print(f"{len(usable)} of {len(comparison)} rewrites passed the validation checks")
    per_play = usable.groupby("creation_id")["delta"].agg(["count", "mean"])
    display(per_play.rename(columns={"count": "passages", "mean": "mean FWED"}))

## Part 5. Run it again

Every answer Gemini gave was saved. Running the same command again asks Gemini nothing: the answers
come from the saved copies, and the numbers are identical. So re-running on the same Google account
costs nothing.

The saved answers are in this account's Drive, so this cell alone does not let someone else, such as
the peer examiner, reproduce the results without a key. For that we plan to publish the recorded
answers and replay them with `--mode replay`; what we promise there is still to be agreed with the
supervisors (agenda item D in `docs/MEETINGS.md`).

In [ ]:
if LIVE_OK:
    replay_dir = SAVE_DIR / "replay" / time.strftime("%Y%m%d-%H%M%S")
    mmlsa(*COMPARE, "--out-dir", replay_dir.as_posix())
    cost_report(replay_dir / "calls.jsonl")
    first = pd.read_csv(live_dir / "comparison.csv")["delta"]
    again = pd.read_csv(replay_dir / "comparison.csv")["delta"]
    print("Same numbers as the first run:", first.equals(again))

## Part 6. The proof of concept (milestone M10)

The pipeline itself, all six steps, on the five creations of `configs/poc.yaml`: two expected
authentic (*Macbeth*, *Hamlet*) and three expected suspicious (*Henry VIII*, *Timon of Athens*,
*A Yorkshire Tragedy*), over `M = 3` runs, each with one foreign creation injected as noise. The
prediction, written down before running: the two expected authentic creations score lower than the
three expected suspicious ones, with no overlap.

It needs about 970 calls, so it needs a paid key. First the plan, which costs nothing: how many calls
the run will make, and a rough cost.

In [ ]:
POC = [
    "run",
    "--config",
    "configs/poc.yaml",
    "--set",
    f"llm.model_id={MODEL}",
    "--set",
    f"run.out_dir={(SAVE_DIR / 'runs').as_posix()}",
    "--set",
    f"llm.cache_dir={CACHE_DIR.as_posix()}",
    *limit_settings(),
]


def plan_number(plan, label):
    # A number from the dry-run plan's line that starts with this label.
    line = next(line for line in plan if line.startswith(label))
    return int(re.search(r"[\d,]+", line[len(label) :]).group().replace(",", ""))


plan = mmlsa(*POC, "--dry-run")
rewrites = plan_number(plan, "rewrite calls")
words = plan_number(plan, "chunk length P")
# A rough upper estimate, with the assumptions of docs/reviews/2026-11-23-colab-notebook-and-cost.md
# (C1, C2): the plan's estimate of input runs 17 per cent low on this corpus, every rewrite prompt
# also carries about 1,500 tokens of profile, a rewrite is as long as its passage at 1.58 tokens per
# word, and 10 per cent of calls are retried.
input_tokens = 1.1 * (1.17 * plan_number(plan, "input tokens") + 1500 * rewrites)
output_tokens = 1.1 * (rewrites * words * 1.58)
cost = dollars(input_tokens, output_tokens)
print(f"\nEstimated tokens: {input_tokens / 1e6:.1f}M in, {output_tokens / 1e6:.1f}M out")
print("Estimated cost:", f"about ${cost:.2f}" if cost is not None else f"no price for {MODEL}")

To run it, set `TIER = "paid"` and `RUN_POC = True` in Part 1, run Part 1 again, then this cell. With
a paid key it takes roughly half an hour.

**If Colab disconnects**, nothing paid for is lost: the run directory and the saved answers are on
Drive. Set `POC_RUN_ID` in Part 1 to the run id this cell printed (it starts with the date and ends
in `-poc`), and run this cell again. The run continues where it stopped, and the answers it already
has come from the saved copies at no cost.

In [ ]:
poc_run = None
if not RUN_POC:
    print("Skipped: set RUN_POC = True in Part 1 to run the proof of concept.")
elif TIER != "paid":
    print("Skipped: the proof of concept needs a paid key and TIER = 'paid' in Part 1.")
elif not HAVE_KEY:
    print("Skipped: no Gemini key. Add GEMINI_API_KEY in Colab Secrets.")
else:
    try:
        mmlsa(*POC, *(["--run-id", POC_RUN_ID] if POC_RUN_ID else []))
    except RuntimeError:
        stopped = SAVE_DIR / "runs" / POC_RUN_ID if POC_RUN_ID else newest_run("poc")
        print(f'\nThe run stopped. To continue it, set POC_RUN_ID = "{stopped.name}" in Part 1.')
        raise
    finally:
        save_cache()
    poc_run = SAVE_DIR / "runs" / POC_RUN_ID if POC_RUN_ID else newest_run("poc")
    cost_report(poc_run / "calls.jsonl")

The result: one score per creation, the threshold, and the chart. Below it, the check of the
prediction.

In [ ]:
EXPECTED_AUTHENTIC = ["the_tragedy_of_macbeth", "the_tragedy_of_hamlet"]  # from configs/poc.yaml

if poc_run is not None:
    poc_scores = show_run(poc_run).set_index("creation_id")["score_mean"]
    authentic = poc_scores[poc_scores.index.isin(EXPECTED_AUTHENTIC)]
    suspicious = poc_scores[~poc_scores.index.isin(EXPECTED_AUTHENTIC)]
    print(f"Highest expected authentic score:  {authentic.max():.4f}")
    print(f"Lowest expected suspicious score:  {suspicious.min():.4f}")
    print("Prediction held (no overlap):", bool(authentic.max() < suspicious.min()))

## Part 7. Is Google Drive fast enough for the saved answers?

Optional, and free: it makes no model calls. The cache writes each answer the way this cell does: a
temporary file, then a rename into place. The cell writes and reads back 200 such files on Drive
and on the Colab disk, deletes them, and prints how long the full corpus run's 9,000 would take. If
Drive is much slower, or if any write fails, set `CACHE_ON_DRIVE = False` in Part 1. Set
`CHECK_DRIVE = True` to run it.

In [ ]:
import tempfile

CHECK_DRIVE = False
FILES = 200


def time_writes(folder):
    # Seconds per file to write via a temporary file and a rename, then read back.
    folder.mkdir(parents=True, exist_ok=True)
    payload = json.dumps({"response": "x" * 2500})  # about the size of one saved answer
    started = time.perf_counter()
    for number in range(FILES):
        handle, temporary = tempfile.mkstemp(dir=folder, suffix=".tmp")
        with os.fdopen(handle, "w", encoding="utf-8") as stream:
            stream.write(payload)
        Path(temporary).replace(folder / f"{number:04d}.json")
    for number in range(FILES):
        (folder / f"{number:04d}.json").read_text(encoding="utf-8")
    seconds = (time.perf_counter() - started) / FILES
    shutil.rmtree(folder)
    return seconds


if CHECK_DRIVE and IN_COLAB:
    for name, folder in [
        ("Drive", SAVE_DIR / "drive_check"),
        ("Colab disk", Path("/content/drive_check")),
    ]:
        seconds = time_writes(folder)
        print(f"{name}: {seconds * 1000:.1f} ms a file, {seconds * 9000 / 60:.1f} min for 9,000")
elif CHECK_DRIVE:
    print("Skipped: this check is about Google Drive, so it runs only in Colab.")

## Where the results are

In Colab they are in Google Drive, under **My Drive → mmlsa**: `runs` (Parts 3 and 6), `live` and
`replay` (Parts 4 and 5), and `cache`, the saved model answers that make Part 5 free.

In [ ]:
for folder in sorted(SAVE_DIR.iterdir()):
    print(folder.name)